# License Plate Detection + Recognition Pipeline

**Stage 1 – Detector:** Ultralytics **YOLOv8n**, fine-tuned on the public Roboflow *License Plate Recognition* dataset (10k images, single class `License_Plate`).
**Stage 2 – Recognizer:** **EasyOCR** on the padded crop, followed by rule-based cleaning for the **Turkish plate format** (`06 ABC 123`).
**Hand-off:** `read_plates(image) → [{box, conf, crop, raw, text, valid}]`. Every box comes from the model's own output; no coordinates are hand-drawn.

**Tool choices**
* **YOLOv8 instead of a Haar cascade:** fine-tuning takes about 15 minutes on a Colab T4, and it handles angled and far-away plates much better than a cascade.
* **EasyOCR as the main OCR engine:** it installs with pip only. Tesseract is installed with `apt-get` inside Colab, which avoids the Windows `tesseract_cmd` path problem; it is used only for the comparison bonus.
* **Colab (Linux + GPU):** it sidesteps both common intern traps, Tesseract paths on Windows and slow CPU training.

> Runtime → Change runtime type → **T4 GPU**, then run the cells top to bottom.

## 0. Install

In [ ]:
!pip -q install ultralytics easyocr roboflow pytesseract
!apt-get -qq install -y tesseract-ocr > /dev/null
import ultralytics, easyocr, cv2, torch
print("ultralytics", ultralytics.__version__, "| GPU:", torch.cuda.is_available())

## 1. Download the training dataset (Roboflow, YOLOv8 format)
Get a free API key at roboflow.com → Settings → API Keys.
Exporting straight to **YOLOv8** format avoids the annotation-format mismatch (VOC XML / COCO JSON vs YOLO txt).

In [ ]:
ROBOFLOW_API_KEY = "PASTE_YOUR_KEY_HERE"

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("roboflow-universe-projects").project("license-plate-recognition-rxg4e")
ds = project.version(11).download("yolov8")   # if v11 errors, try another version number from the dataset page
print(ds.location)

Roboflow's `data.yaml` uses relative paths such as `../train/images`, which Ultralytics often resolves wrongly (*"images not found"*).
This cell rewrites them as absolute paths.

In [ ]:
import yaml, os
yp = f"{ds.location}/data.yaml"
cfg = yaml.safe_load(open(yp))
for split, folder in [("train", "train"), ("val", "valid"), ("test", "test")]:
    p = f"{ds.location}/{folder}/images"
    if os.path.isdir(p):
        cfg[split] = p
    elif split in cfg:
        del cfg[split]
yaml.safe_dump(cfg, open(yp, "w"))
print(open(yp).read())

## 2. Fine-tune YOLOv8n  (~15–25 min on a T4)

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")                       # COCO-pretrained start point
model.train(data=yp, epochs=30, imgsz=640, batch=16, patience=10, name="plates")
BEST = model.trainer.best                         # path to best.pt
print("best weights:", BEST)

In [ ]:
# Validation metrics on the dataset's own val split (report mAP50 / mAP50-95 in the README)
det = YOLO(BEST)
m = det.val(data=yp, split="val", verbose=False)
print(f"mAP50 = {m.box.map50:.3f} | mAP50-95 = {m.box.map:.3f}")

In [ ]:
# Save weights to Drive so a Colab disconnect doesn't lose them
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/plate_pipeline
!cp {BEST} /content/drive/MyDrive/plate_pipeline/best.pt
# Later sessions: det = YOLO("/content/drive/MyDrive/plate_pipeline/best.pt")

## 3. Pipeline module (detector → crop → OCR → cleaning)

In [ ]:
%%writefile pipeline.py
"""
License plate pipeline: YOLOv8 detector -> padded crop -> OCR -> rule-based cleaning.

Main entry point:
    read_plates(image, model, reader=None, engine="easyocr") -> list of dicts
        box   : (x1, y1, x2, y2) from the MODEL's own output (no manual cropping)
        conf  : detector confidence
        crop  : padded plate crop (BGR numpy array)
        raw   : raw OCR text
        text  : cleaned plate text (Turkish format rules applied)
        valid : True if the cleaned text matches the Turkish plate regex
"""
import re
import cv2
import numpy as np

# ---------------------------------------------------------------- plate format
# Turkish plate = 2-digit province code (01-81) + letters + digits:
#   1 letter + 4 digits | 2 letters + 3-4 digits | 3 letters + 2-3 digits
PLATE_REGEX = re.compile(
    r"^(0[1-9]|[1-7][0-9]|8[01])([A-Z][0-9]{4}|[A-Z]{2}[0-9]{3,4}|[A-Z]{3}[0-9]{2,3})$"
)
ALNUM = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"

# Common OCR confusions, applied only where the plate format says which kind of
# character must be at that position.
TO_DIGIT = {"O": "0", "Q": "0", "D": "0", "I": "1", "L": "1", "Z": "2",
            "S": "5", "B": "8", "G": "6"}
TO_LETTER = {"0": "O", "1": "I", "2": "Z", "5": "S", "8": "B", "6": "G", "4": "A"}


def compact(s):
    """Uppercase and keep only A-Z / 0-9 (used for both predictions and ground truth)."""
    return re.sub(r"[^A-Z0-9]", "", str(s).upper())


def _fix(segment, table, want):
    """Force every char in segment to be a digit or a letter. Returns (text, n_substitutions)."""
    out, subs = [], 0
    for ch in segment:
        ok = ch.isdigit() if want == "digit" else ch.isalpha()
        if ok:
            out.append(ch)
        elif ch in table:
            out.append(table[ch])
            subs += 1
        else:
            return None, 99
    return "".join(out), subs


def clean_plate(raw):
    """
    Rule-based cleaning for Turkish plates.
    1. Strip spaces/symbols, uppercase.
    2. Drop the 'TR' country code printed on the blue EU strip.
    3. Positions 1-2 must be digits (O->0, I->1, B->8 ...).
    4. Try every legal letters/digits split; in the letter block fix 0->O, 1->I, 8->B ...,
       in the digit block fix O->0 etc. Keep the valid candidate needing the fewest fixes.
    Returns (clean_text, is_valid_format).
    """
    s = compact(raw)
    if s.startswith("TR") and len(s) > 7:
        s = s[2:]
    if len(s) < 6:
        return s, False

    province, p_subs = _fix(s[:2], TO_DIGIT, "digit")
    if province is None:
        return s, False

    best = None
    rest = s[2:]
    for n_letters in (1, 2, 3):
        letters, l_subs = _fix(rest[:n_letters], TO_LETTER, "letter")
        digits, d_subs = _fix(rest[n_letters:], TO_DIGIT, "digit")
        if letters is None or digits is None:
            continue
        cand = province + letters + digits
        if PLATE_REGEX.match(cand):
            score = p_subs + l_subs + d_subs
            if best is None or score < best[1]:
                best = (cand, score)
    if best:
        return best[0], True
    return s, False


def format_plate(s):
    """'06ABC123' -> '06 ABC 123' for display."""
    m = re.match(r"^(\d{2})([A-Z]+)(\d+)$", s)
    return f"{m.group(1)} {m.group(2)} {m.group(3)}" if m else s


# ---------------------------------------------------------------- detection
def detect_plates(img, model, conf=0.25):
    """Run the YOLO model. Returns [(box_xyxy, conf), ...] sorted by confidence. [] if none."""
    r = model.predict(img, conf=conf, verbose=False)[0]
    if r.boxes is None or len(r.boxes) == 0:
        return []
    xyxy = r.boxes.xyxy.cpu().numpy()
    cf = r.boxes.conf.cpu().numpy()
    dets = [(tuple(int(v) for v in b), float(c)) for b, c in zip(xyxy, cf)]
    return sorted(dets, key=lambda d: -d[1])


def crop_with_padding(img, box, pad=0.10):
    """Crop box + pad% on each side, clamped to image borders."""
    h, w = img.shape[:2]
    x1, y1, x2, y2 = box
    px, py = int((x2 - x1) * pad), int((y2 - y1) * pad)
    x1, y1 = max(0, x1 - px), max(0, y1 - py)
    x2, y2 = min(w, x2 + px), min(h, y2 + py)
    return img[y1:y2, x1:x2].copy(), (x1, y1, x2, y2)


# ---------------------------------------------------------------- OCR
def preprocess(crop):
    """Grayscale, upscale small plates to ~100px tall, denoise while keeping edges."""
    g = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    scale = max(1.0, 100 / max(g.shape[0], 1))
    g = cv2.resize(g, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    return cv2.bilateralFilter(g, 9, 75, 75)


def run_ocr(crop, engine="easyocr", reader=None):
    g = preprocess(crop)
    if engine == "easyocr":
        res = reader.readtext(g, allowlist=ALNUM, detail=1, paragraph=False)
        res = sorted(res, key=lambda r: min(p[0] for p in r[0]))  # left -> right
        return " ".join(r[1] for r in res)
    if engine == "tesseract":
        import pytesseract
        _, b = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        cfg = f"--psm 7 -c tessedit_char_whitelist={ALNUM}"
        return pytesseract.image_to_string(b, config=cfg).strip()
    raise ValueError(f"unknown engine {engine}")


# ---------------------------------------------------------------- pipeline
def read_plates(image, model, reader=None, engine="easyocr", conf=0.25, pad=0.10):
    """Image (path or BGR array) in -> list of plate results out. Handles 0 or many plates."""
    img = cv2.imread(image) if isinstance(image, str) else image
    if img is None:
        raise ValueError(f"could not read image: {image}")
    results = []
    for box, c in detect_plates(img, model, conf):
        crop, _ = crop_with_padding(img, box, pad)
        if crop.size == 0:
            continue
        raw = run_ocr(crop, engine, reader)
        text, valid = clean_plate(raw)
        results.append(dict(box=box, conf=c, crop=crop, raw=raw, text=text, valid=valid))
    return results


def draw_results(img, results, gt_boxes=None):
    out = img.copy()
    for g in gt_boxes or []:
        cv2.rectangle(out, g[:2], g[2:], (0, 200, 0), 2)  # green = ground truth
    for r in results:
        x1, y1, x2, y2 = r["box"]
        cv2.rectangle(out, (x1, y1), (x2, y2), (0, 0, 255), 2)  # red = prediction
        label = f"{format_plate(r['text'])} {r['conf']:.2f}"
        cv2.putText(out, label, (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX,
                    0.7, (0, 0, 255), 2)
    return out


# ---------------------------------------------------------------- metrics
def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def levenshtein(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


def cer(pred, gt):
    """Character error rate = edit distance / length of ground truth."""
    gt = compact(gt)
    return levenshtein(compact(pred), gt) / max(len(gt), 1)

In [ ]:
import importlib, pipeline; importlib.reload(pipeline)
from pipeline import *
reader = easyocr.Reader(["en"], gpu=torch.cuda.is_available())

# Unit checks for the cleaning rules (0/O, 1/I, 8/B by position)
for raw in ["06 ABC 123", "O6 A8C I23", "TR 34 AB 12O4", "35 B 1234", "06 AB 12"]:
    print(f"{raw:15s} -> {clean_plate(raw)}")

## 4. Your hand-labeled test set (≥ 20 images)
1. Photograph 20–30 cars and include **at least 3 photos with several cars** in frame, plus some angled, blurry, or far-away plates.
2. Create a new Roboflow project (Object Detection, class `plate`), upload the photos, and draw a box on every readable plate.
3. Generate a version. **Remove the "Resize" preprocessing step** (keep Auto-Orient), then Export → **YOLOv8** → download the zip.
4. Upload the zip here as `test_set.zip`.

Roboflow renames files to `IMG_1234_jpg.rf.<hash>.jpg`. The code below maps them back to `IMG_1234` so the names match your CSV.

In [ ]:
from google.colab import files
up = files.upload()          # choose test_set.zip
!rm -rf /content/test && mkdir -p /content/test && unzip -q -o test_set.zip -d /content/test

In [ ]:
import glob, re, pandas as pd

def base_name(path):
    stem = os.path.splitext(os.path.basename(path))[0]
    return re.sub(r"_(jpe?g|png|heic|webp)\.rf\.[0-9a-f]+$", "", stem, flags=re.I).lower()

def load_yolo_labels(img_path):
    """YOLO txt (class cx cy w h, normalized) -> list of pixel boxes (x1,y1,x2,y2)."""
    lbl = re.sub(r"/images/", "/labels/", os.path.splitext(img_path)[0] + ".txt")
    img = cv2.imread(img_path); h, w = img.shape[:2]
    boxes = []
    if os.path.exists(lbl):
        for line in open(lbl):
            p = line.split()
            if len(p) < 5: continue
            cx, cy, bw, bh = map(float, p[1:5])
            boxes.append((int((cx-bw/2)*w), int((cy-bh/2)*h), int((cx+bw/2)*w), int((cy+bh/2)*h)))
    return boxes

test_imgs = sorted(p for p in glob.glob("/content/test/**/images/*", recursive=True)
                   if p.lower().endswith((".jpg", ".jpeg", ".png")))
GT_BOXES = {p: load_yolo_labels(p) for p in test_imgs}
print(len(test_imgs), "test images,", sum(map(len, GT_BOXES.values())), "labeled plates")
assert len(test_imgs) >= 20, "brief requires at least 20 hand-labeled test images"

### Ground-truth plate strings
The first run writes `ground_truth.csv` with one row per labeled box, in left-to-right order.
Download it, type the real plate into the `plate` column (e.g. `06 ABC 123`), and upload it again.

In [ ]:
GT_CSV = "/content/ground_truth.csv"
if not os.path.exists(GT_CSV):
    rows = []
    for p in test_imgs:
        for i, b in enumerate(sorted(GT_BOXES[p], key=lambda b: b[0])):
            rows.append({"image": base_name(p), "plate_idx": i, "x1": b[0], "plate": ""})
    pd.DataFrame(rows).to_csv(GT_CSV, index=False)
    files.download(GT_CSV)
    print("Fill in the 'plate' column, then run the next cell to upload it.")

In [ ]:
up = files.upload()                                   # upload the filled ground_truth.csv
!cp "{list(up)[0]}" {GT_CSV}
gt_df = pd.read_csv(GT_CSV, dtype={"plate": str}).fillna("")
GT_TEXT = {}                                          # image -> list of plate strings, left-to-right
for name, g in gt_df.sort_values(["image", "plate_idx"]).groupby("image"):
    GT_TEXT[name] = list(g["plate"])
print(gt_df.head(10)); print("empty plates:", (gt_df.plate == "").sum())

## 5. Run the pipeline and evaluate
* **Detection:** each GT box is greedily matched to the unused prediction with the highest IoU. An unmatched GT box scores IoU = 0 and counts as a miss.
  Reported: **mean IoU**, recall@0.5, and precision@0.5.
* **OCR:** the cleaned text of the matched prediction is compared with the GT string. Reported: **exact match** and **CER** (edit distance / GT length).
  A missed plate counts as wrong (CER = 1), so the accuracy is **end-to-end**.

In [ ]:
def evaluate(engine="easyocr", conf=0.25, verbose=True):
    rows, per_image = [], {}
    for p in test_imgs:
        name = base_name(p)
        preds = read_plates(p, det, reader, engine=engine, conf=conf)
        per_image[p] = preds
        gts = sorted(GT_BOXES[p], key=lambda b: b[0])
        texts = GT_TEXT.get(name, [""] * len(gts))
        used = set()
        for gi, g in enumerate(gts):
            best_j, best_iou = None, 0.0
            for j, pr in enumerate(preds):
                if j in used: continue
                v = iou(g, pr["box"])
                if v > best_iou: best_j, best_iou = j, v
            if best_j is not None and best_iou > 0: used.add(best_j)
            pr = preds[best_j] if best_j is not None and best_iou > 0 else None
            gt_txt = texts[gi] if gi < len(texts) else ""
            pred_txt = pr["text"] if pr else ""
            rows.append(dict(image=name, gt_box=g, pred_box=pr["box"] if pr else None,
                             conf=round(pr["conf"], 2) if pr else None, IoU=round(best_iou, 3),
                             raw_ocr=pr["raw"] if pr else "", clean=format_plate(pred_txt),
                             gt_plate=gt_txt, valid_fmt=pr["valid"] if pr else False,
                             CER=round(cer(pred_txt, gt_txt), 3),
                             correct=(compact(pred_txt) == compact(gt_txt) and gt_txt != "")))
        for j, pr in enumerate(preds):                # false positives
            if j not in used:
                rows.append(dict(image=name, gt_box=None, pred_box=pr["box"], conf=round(pr["conf"], 2),
                                 IoU=0.0, raw_ocr=pr["raw"], clean=format_plate(pr["text"]),
                                 gt_plate="(no plate)", valid_fmt=pr["valid"], CER=None, correct=False))
    df = pd.DataFrame(rows)
    real = df[df.gt_box.notna()]
    n_pred = sum(len(v) for v in per_image.values())
    s = dict(engine=engine, plates=len(real),
             mean_IoU=real.IoU.mean(), recall_50=(real.IoU >= 0.5).mean(),
             precision_50=(real.IoU >= 0.5).sum() / max(n_pred, 1),
             exact_match=real.correct.mean(), mean_CER=real.CER.mean())
    if verbose:
        print(" | ".join(f"{k}={v:.3f}" if isinstance(v, float) else f"{k}={v}" for k, v in s.items()))
    return df, s, per_image

report, summary, PREDS = evaluate("easyocr")

In [ ]:
# End-to-end accuracy report: image -> detected box -> OCR text -> correct/incorrect
pd.set_option("display.max_rows", 200, "display.width", 250)
show = report.copy(); show["correct"] = show.correct.map({True: "✅", False: "❌"})
display(show[["image", "pred_box", "conf", "IoU", "raw_ocr", "clean", "gt_plate", "CER", "correct"]])
print(f"\nMean IoU: {summary['mean_IoU']:.3f}   |   End-to-end exact-match accuracy: "
      f"{summary['exact_match']*100:.1f}%   |   Mean CER: {summary['mean_CER']:.3f}")
report.to_csv("accuracy_report.csv", index=False)

## 6. Multi-vehicle cropping check (≥ 3 images with several plates)

In [ ]:
import matplotlib.pyplot as plt
multi = [p for p in test_imgs if len(GT_BOXES[p]) >= 2]
print(len(multi), "multi-plate images")
for p in multi[:4]:
    img = cv2.imread(p); preds = PREDS[p]
    n = max(len(preds), 1)
    fig, ax = plt.subplots(1, n + 1, figsize=(5 + 3 * n, 4), gridspec_kw={"width_ratios": [3] + [1] * n})
    ax = np.atleast_1d(ax)
    ax[0].imshow(cv2.cvtColor(draw_results(img, preds, GT_BOXES[p]), cv2.COLOR_BGR2RGB))
    ax[0].set_title(f"{base_name(p)}: {len(GT_BOXES[p])} GT (green) / {len(preds)} pred (red)")
    for i, r in enumerate(preds):
        ax[i + 1].imshow(cv2.cvtColor(r["crop"], cv2.COLOR_BGR2RGB))
        ax[i + 1].set_title(format_plate(r["text"]), fontsize=10)
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.show()

## 7. Failure case analysis
Each failure is assigned to the stage that caused it, and simple image measurements show why:
* **Detector** failures: a miss (IoU < 0.5) or a false positive.
* **OCR** failures: the box was right but the text was wrong. The measurements are
  **blur** (Laplacian variance < 100), **small plate** (crop height < 25 px), **angle** (aspect ratio far from a flat EU plate's ~4.5:1),
  and **low contrast / glare**.

In [ ]:
def diagnose(row, img):
    if row.gt_box is None: return "DETECTOR – false positive (non-plate region detected)"
    if row.IoU < 0.5:     return "DETECTOR – plate missed or badly localized (IoU < 0.5)"
    x1, y1, x2, y2 = row.pred_box
    crop = cv2.cvtColor(img[y1:y2, x1:x2], cv2.COLOR_BGR2GRAY)
    reasons = []
    blur = cv2.Laplacian(crop, cv2.CV_64F).var()
    if blur < 100: reasons.append(f"blurry (Laplacian var {blur:.0f})")
    if crop.shape[0] < 25: reasons.append(f"small/far plate ({crop.shape[0]}px tall)")
    ar = crop.shape[1] / max(crop.shape[0], 1)
    if ar < 3.0: reasons.append(f"angled/perspective or 2-row plate (aspect {ar:.1f})")
    if crop.std() < 35: reasons.append(f"low contrast/glare (std {crop.std():.0f})")
    if not row.valid_fmt: reasons.append("OCR output doesn't fit TR format (extra/missing chars)")
    return "OCR – " + (", ".join(reasons) or "character confusion despite a clean crop")

fails = report[~report.correct].copy()
paths = {base_name(p): p for p in test_imgs}
fails["diagnosis"] = [diagnose(r, cv2.imread(paths[r.image])) for r in fails.itertuples()]
display(fails[["image", "IoU", "raw_ocr", "clean", "gt_plate", "diagnosis"]])

for r in list(fails.itertuples())[:5]:
    img = cv2.imread(paths[r.image])
    fig, ax = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [3, 1]})
    ax[0].imshow(cv2.cvtColor(draw_results(img, PREDS[paths[r.image]], GT_BOXES[paths[r.image]]), cv2.COLOR_BGR2RGB))
    ax[0].set_title(f"{r.image}  |  GT: {r.gt_plate}  |  pred: {r.clean or '—'}")
    box = r.pred_box or r.gt_box
    ax[1].imshow(cv2.cvtColor(img[box[1]:box[3], box[0]:box[2]], cv2.COLOR_BGR2RGB))
    ax[1].set_title("pred crop" if r.pred_box else "GT region (missed)")
    for a in ax: a.axis("off")
    fig.suptitle(r.diagnosis, color="crimson"); plt.tight_layout(); plt.show()

**Write-up (edit after running):** for at least 3 failures above, say which stage failed and why. For example:
1. `IMG_xxxx`: *detector miss*. The plate is at a steep angle, about 20 px tall, and partly behind another car's bumper.
2. `IMG_yyyy`: *OCR*. Motion blur merged `8` and `B`, and the position rule fixed only one of the two.
3. `IMG_zzzz`: *OCR*. Glare over the middle letters, so EasyOCR returned 2 characters instead of 3.

## 8. Bonus – Tesseract vs EasyOCR (same detector, same test set)

In [ ]:
rep_t, sum_t, _ = evaluate("tesseract")
cmp = pd.DataFrame([summary, sum_t]).set_index("engine")[["plates", "mean_IoU", "exact_match", "mean_CER"]]
display(cmp.round(3))

## 9. Bonus – Parking log (SQLite)

In [ ]:
import sqlite3, datetime
con = sqlite3.connect("parking_log.db")
con.execute("CREATE TABLE IF NOT EXISTS log (ts TEXT, plate TEXT, valid INTEGER, conf REAL, source TEXT)")

def log_image(path):
    for r in read_plates(path, det, reader):
        con.execute("INSERT INTO log VALUES (?,?,?,?,?)",
                    (datetime.datetime.now().isoformat(timespec="seconds"), format_plate(r["text"]),
                     int(r["valid"]), round(r["conf"], 3), os.path.basename(path)))
    con.commit()

for p in test_imgs[:10]: log_image(p)
display(pd.read_sql("SELECT * FROM log ORDER BY ts DESC", con))

## 10. Bonus – Dashcam video with live boxes

In [ ]:
up = files.upload()                           # a short dashcam .mp4 (10–30 s is plenty)
src = list(up)[0]
cap = cv2.VideoCapture(src)
fps = cap.get(cv2.CAP_PROP_FPS) or 25
W, H = int(cap.get(3)), int(cap.get(4))
out = cv2.VideoWriter("annotated.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
last, i = [], 0
while True:
    ok, frame = cap.read()
    if not ok: break
    if i % 5 == 0:                            # OCR every 5th frame, reuse boxes in between
        last = read_plates(frame, det, reader)
    out.write(draw_results(frame, last)); i += 1
cap.release(); out.release()
print("frames:", i); files.download("annotated.mp4")

## 11. Anti-cheat demo: 3 unseen images
The mentor uploads 3 new images and the pipeline runs on them unchanged. The boxes come only from `det`, the fine-tuned model.

In [ ]:
up = files.upload()
for name in up:
    res = read_plates(name, det, reader)
    img = cv2.imread(name)
    plt.figure(figsize=(10, 6)); plt.axis("off")
    plt.imshow(cv2.cvtColor(draw_results(img, res), cv2.COLOR_BGR2RGB))
    plt.title(f"{name}: " + (", ".join(format_plate(r['text']) for r in res) or "no plate detected")); plt.show()
    for r in res: print(f"  box={r['box']} conf={r['conf']:.2f} raw='{r['raw']}' -> {format_plate(r['text'])} valid={r['valid']}")